# 01 — Quick Start: AryColBring in 5 Minutes

**Objective**: go from raw interaction data to a top-N recommendation list, end to end, in the shortest path possible.

**Audience**: new users who want to see the shape of the whole pipeline before diving into the details covered in the other notebooks.

> **A note on running this notebook**: `AryColBringModelTrainer`/`TheReasoner`'s actual scoring path is backed by a compiled Cython extension (`CLproximity`), and importing anything under `src.*` also pulls in `duckdb` transitively. Both are declared dependencies (`requirements.txt` / `pyproject.toml`) and will be present in a normal project environment — but to keep this notebook runnable even in a bare environment (e.g. for review), Section 3 below uses a small **synthetic stand-in** that reproduces the real scoring math (dot product of user/item embeddings + biases — see `AryColBringPredictor.predict()`), and Section 4 attempts the real dashboard import with a graceful fallback.

## 1. Setup

In [ ]:
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd

DATA_DIR  = Path("./data")
MODEL_DIR = Path("./artifacts")
np.random.seed(0)
print("Setup OK.")

## 2. The real pipeline (for reference)

In a real project environment, the quick-start path is exactly two classes:

```python
from src.models.arycolbring.trainer   import AryColBringModelTrainer
from src.models.arycolbring.inference import AryColBringInference

# --- Train ---
trainer = AryColBringModelTrainer(no_components=16, loss="warp",
                                  learning_schedule="adagrad", random_state=42)
trainer.fit(interactions)                 # interactions: scipy.sparse matrix (users x items)
trainer.save_model(MODEL_DIR / "ACBmodel" / "model.npz")

# --- Serve ---
infer = AryColBringInference(model_path=MODEL_DIR / "ACBmodel" / "model.npz",
                             num_threads=4, cache_enabled=True)
recommendations = infer.recommend(user_id=0, n_items=10, exclude_items=[3, 7])
# -> [(item_id, score), ...] sorted by score descending
```

See `02_Data_Preparation.ipynb` for how `interactions` is actually built from raw transaction data, and `03_Training_AryColBring.ipynb` for the full training/evaluation loop.

## 3. Runnable mini-demo (synthetic embeddings)

`AryColBringPredictor.predict()` scores a (user, item) pair as `dot(user_embedding, item_embedding) + user_bias + item_bias` (see `src/models/arycolbring/inout/approximator.py`). We reproduce exactly that here with random embeddings, so the *shape* of `recommend()`'s output and the rest of this notebook are faithful to the real API even without a trained model.

In [ ]:
N_USERS, N_ITEMS, DIM = 200, 60, 16

user_embeddings = np.random.default_rng(0).normal(size=(N_USERS, DIM))
item_embeddings = np.random.default_rng(1).normal(size=(N_ITEMS, DIM))
user_biases     = np.random.default_rng(2).normal(scale=0.1, size=N_USERS)
item_biases     = np.random.default_rng(3).normal(scale=0.1, size=N_ITEMS)


def mock_recommend(user_id: int, n_items: int = 10,
                    exclude_items: List[int] = None) -> List[Tuple[int, float]]:
    """Stands in for AryColBringInference.recommend() -- same scoring
    formula (dot product + biases), same return shape."""
    exclude_items = set(exclude_items or [])
    candidates = [i for i in range(N_ITEMS) if i not in exclude_items]
    scores = (item_embeddings[candidates] @ user_embeddings[user_id]
              + item_biases[candidates] + user_biases[user_id])
    order = np.argsort(scores)[::-1][:n_items]
    return [(candidates[idx], float(scores[idx])) for idx in order]


recs = mock_recommend(user_id=0, n_items=10, exclude_items=[3, 7])
for rank, (item_id, score) in enumerate(recs, start=1):
    print(f"{rank:>2}. item {item_id:>3}  score={score:.4f}")

## 4. Visualizing the result on the inference dashboard

The recommendations dashboard (see `src/models/arycolbring/narative/`) is pure Python/Jinja2 -- no Cython involved -- so we can feed it the mock output directly. This cell attempts the real import; if `duckdb` (a transitive import of `src.*`) isn't installed in *this* environment, it prints a clear note instead of failing the whole notebook.

In [ ]:
try:
    from src.models.arycolbring.narative.rearender import build_inference_context
    from src.assets import score_distribution

    predictions = [{"user_id": 0, "item_id": iid, "score": sc, "rank": r + 1}
                   for r, (iid, sc) in enumerate(recs)]
    context = build_inference_context({
        "metrics": {"avg_latency_ms": 8.2, "qps": 900},
        "inference_statistics": {"n_predictions": len(recs), "n_users_served": 1,
                                 "avg_latency_ms": 8.2, "throughput_preds_per_sec": 900.0},
        "predictions": predictions,
        "item_embeddings": item_embeddings,
        "item_ids": list(range(N_ITEMS)),
        "embeddings": {"vectors": item_embeddings, "ids": list(range(N_ITEMS))},
    })
    print("Dashboard context built OK.")
    print("Gauges (should be empty -- no eval metrics in this report):", context["gauges"])
    print("Score distribution n =", context["score_distribution"]["n"])
except ImportError as exc:
    print(f"Skipping real dashboard import in this environment ({exc}).")
    print("See 04_Interactive_Dashboard.ipynb for the full walkthrough.")

## Summary

- `AryColBringModelTrainer.fit()` + `.save_model()` trains and persists a model.
- `AryColBringInference(model_path=...).recommend(user_id, n_items)` serves top-N recommendations, scored as `dot(user_emb, item_emb) + biases`.
- The inference dashboard (`narative/rearender.py`) consumes that same `predictions` shape directly -- no glue code needed.

**Next**: `02_Data_Preparation.ipynb` for turning raw transactions into the `interactions` matrix this pipeline expects.